# 🕹️ <span style=color:dodgerblue>LLM for video game knowledge assistance<span>

This notebook is used for explanation purpose. Once the `docker compose up` command
is executed the entire application is ready to be used.

## 📔 <span style=color:gold>What is this notebook about?</span>
This notebook is intended to explain all the pipeline and provide an overview 
of the processes involved in the application (ingestion, monitoring, evaluation, etc).  
It's a complement to the [README.md](README.md) that shows the internal mechanisms
of the scripts.

In [1]:
from llm import RAGClient
from opensearchpy import OpenSearch

## <span style=color:green>📂 Opensearch client creation</span>

We create an opensearch client. It is used for indexation and search 
(lexical, semantic, hybrid).  
The `RAGClient` simplifies the use of the RAG.

In [2]:
# Opensearch client connection to the running docker container.
opensearch_client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "Opensearch16admin#"),
    use_ssl=False,
    verify_certs=False,
    ssl_show_warn=False,
)

# This client has a large set of functionalities that ease the use of the llm.
rag_client = RAGClient(
    opensearch_client,
    model="gemini-3.1-flash-lite", # This is usually the best free model.
)

## <span style=color:lightsalmon>⛁ Ingestion</span>

This part showcases the ingestion process and how It works.  
To avoid the long waiting time this part is entirely optional and is not a pre-requisite to advance
to the other blocks of this notebook.  
It's just used for explanation purpose.

Before anything we call the helper function `setup_embedder` this will set the 
ML model for our vector/semantic search (and hybrid search).  
Note that the `setup_embedder` will return the `model_id` that was already set up when we called
`docker compose up`.

In [3]:
# Helper for downloading and preparing the ML embedder (convert text to vector).
from opensearch_utils import setup_embedder

In [4]:
# This will give us the model id for future use.
model_id = setup_embedder(opensearch_client) 

Found existing model ID 'S9xdtJ8Bz4lTtz-qdxKD' in state: DEPLOYED
✅ Model 'S9xdtJ8Bz4lTtz-qdxKD' is active and deployed. Reusing it.


### 🎮 <span style=color:darkorchid>IGDB ingestion</span>

**You can skip this section if you want.**

> <span style=color:gold>⚠️ **Warning**</span>    
> If you want to execute the IGDB ingestion yourself, you must have and IGDB 
> developer key and an account.  
> Please follow the instructions in this link: https://api-docs.igdb.com/#getting-started.  
> Then write all your information in the `.env` file.

In [5]:
# We import this two helpers for ingestion
from ingest import IGDB, Wikipedia

In [6]:
igdb = IGDB(opensearch_client)
# igdb.download(index="igdb_small")

### <span style=color:deepskyblue>📄 Wikipedia ingestion</span>

**You can skip this section if you want.**

> For the wikipedia ingest, **you don't need to do any extra setup** (even if you 
> skipped igdb ingestion part).


We pull the wikipedia information from huggingface index. The data was updated 
on 


In [7]:
wikipedia = Wikipedia(opensearch_client)
# wikipedia.download(index="wikipedia_small")

## 🧪 <span style=color:forestgreen>Evaluation</span>

Here we will use our `Evaluator` class to execute each one of the steps:
1. Ground truth generation
2. Search evaluation & optimization.
    1. Perform the base evaluation itself.
    2. Optimize the boost values.
3. Tools use and final RAG's answer evaluation.

In [8]:
import pandas as pd
from evaluation import Evaluator

# Free llm model for the judge
evaluator = Evaluator(RAGClient(opensearch_client, model="gemma-4-31b-it")) 

### 🎯 <span style=color:orangered>Ground truth generation</span>

In order to evaluate the search quality and model's performance we must have 
querys and a target variable to use as evaluation method.  
In this case, the target variable is the document id and the query will be a llm 
generated question based on the target document. 

For instance, if the target document is about Mario Kart, the llm
will generate `questions_per_doc` questions related to the document topic.

> <span style=color:yellow>⚠️ **Warning**</span>  
> The code blocks in this section create a very small dataset so as to
> not waste your tokens.  
> You may create a very large dataset if you want.  
> Keep in mind that there is a pre-generated large ground truth dataset, 
> so you don't need to create one from zero.

This should take $\sim 2\text{-}3 \ \text{min}$ 

In [9]:
# wikipedia_ground_truth_small = evaluator.generate_ground_truth(
#     index="wikipedia",
#     questions_per_doc=3,
#     num_docs=6,
#     file_path="data/wikipedia_ground_truth_small.csv",
# )

# igdb_ground_truth_small = evaluator.generate_ground_truth(
#     index="igdb",
#     questions_per_doc=3,
#     num_docs=6,
#     file_path="data/igdb_ground_truth_small.csv",
# )

### 🔎 <span style=color:goldenrod>Search evaluation & optimization</span>

We must evaluate our search functions. Do they return the relevant documents?
For this we will use two metrics

If you use the full dataset this will take around $10$ min, you can bring a coffe ☕

In [10]:
evaluator.ground_truth = pd.read_csv("data/igdb_ground_truth_small.csv")
igdb_hr_score, igdb_mrr_score, igdb_x = evaluator.evaluate_search(index="igdb",num=5)

evaluator.ground_truth = pd.read_csv("data/wikipedia_ground_truth_small.csv")
wikipedia_hr_score, wikipedia_mrr_score, wikipedia_x = evaluator.evaluate_search(
    index="wikipedia",num=5
)

print(
    "--- IGDB search evaluation ---",
    f"Hit rate score: {igdb_hr_score}",
    f"Mean reciprocal rank score: {igdb_mrr_score}",
    "--- wikipedia search evaluation ---",
    f"Hit rate score: {wikipedia_hr_score}",
    f"Mean reciprocal rank score: {wikipedia_mrr_score}", sep="\n"
)

Evaluating Search:   0%|          | 0/24 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 24/24 [00:00<00:00, 209.47it/s]

--- IGDB search evaluation ---
Hit rate score: 0.7916666666666666
Mean reciprocal rank score: 0.7270833333333333
--- wikipedia search evaluation ---
Hit rate score: 1.0
Mean reciprocal rank score: 0.8902777777777778


Now, let's test for the bigger dataset.

In [ ]:
num = 5  # Number of retrieved documents per search

evaluator.ground_truth = pd.read_csv("data/igdb_ground_truth.csv")
igdb_hr_score, igdb_mrr_score, igdb_x = evaluator.evaluate_search(
    index="igdb",
    search_type="hybrid",
    num=num,
)

# import time

# time.sleep(5)
evaluator.ground_truth = pd.read_csv("data/wikipedia_ground_truth.csv")
wikipedia_hr_score, wikipedia_mrr_score, wikipedia_x = evaluator.evaluate_search(
    index="wikipedia",
    search_type="hybrid",
    num=num,
)

print(
    "--- IGDB search evaluation ---",
    f"Hit rate score: {igdb_hr_score}",
    f"Mean reciprocal rank score: {igdb_mrr_score}",
    "--- wikipedia search evaluation ---",
    f"Hit rate score: {wikipedia_hr_score}",
    f"Mean reciprocal rank score: {wikipedia_mrr_score}",
    sep="\n",
)

Evaluating Search: 100%|██████████| 60/60 [00:00<00:00, 66.84it/s]

--- IGDB search evaluation ---
Hit rate score: 0.8035516093229744
Mean reciprocal rank score: 0.6957269700332963
--- wikipedia search evaluation ---
Hit rate score: 0.7833333333333333
Mean reciprocal rank score: 0.7174999999999999


Now we want to use a `boost_dict` which is a way to increase or decrease the 
importance/relevance of different search fields. For instance we may set a boost of $2$
for the *name* field and $3$ for the *storyline*, so that the search will tend to prioritize
word match more in the *storyline* field and less in the *name* field.

Here our optimization method is pure brute force, we will only test in a given set of values (n dimensional grid) and take the combination that returns the best result. It's simply that.

We will perform the optimization on each index (wikipedia and igdb) separately.
Given that this takes a long time we will just test a very small set of possible values. 

In [ ]:
# performance_df = pd.DataFrame()
results = []
from itertools import product

# Search boosting optimization
# For IGDB index
import time

evaluator.ground_truth = pd.read_csv("data/igdb_ground_truth.csv")
vals = (0, 1, 2, 3)
iter = 0
for i, j, k, search_type in product(vals, vals, vals, ("lexical", "hybrid")):
    if i == j == k != 1:
        continue

    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="igdb",
        search_type=search_type,
        boost_dict={"name": i, "summary": j, "storyline": k},
    )

    results.append(
        {
            "name": i,
            "summary": j,
            "storyline": k,
            "hr": hr_score,
            "mrr": mrr_score,
            "type": search_type,
        }
    )

    if iter % 5 == 0:
        time.sleep(2.0)

    iter += 1

igdb_performance_df = pd.DataFrame(results)

# For Wikipedia index

Evaluating Search: 100%|██████████| 901/901 [00:37<00:00, 24.34it/s]


In [ ]:
df = igdb_performance_df.sort_values("mrr", ascending=False)
print(df[df.type == "lexical"].head(1))
print()
print(df[df.type == "hybrid"].head(1))

    name  summary  storyline        hr       mrr     type
26     1        1          1  0.843507  0.735313  lexical

    name  summary  storyline        hr       mrr    type
27     1        1          1  0.803552  0.695727  hybrid


In [ ]:
evaluator.ground_truth = pd.read_csv("data/wikipedia_ground_truth.csv")
del results
results = []
vals = (0, 1, 2, 3)
iter = 0
for i, j, search_type in product(vals, vals, ("lexical", "hybrid")):
    if i == j == k != 1:
            continue
    
    hr_score, mrr_score, _ = evaluator.evaluate_search(
        index="wikipedia",
        search_type=search_type,
        boost_dict={"title": i, "text": j},
    )

    results.append(
        {
            "title": i,
            "text": j,
            "hr": hr_score,
            "mrr": mrr_score,
            "type": search_type,
        }
    )

    if iter % 5 == 0:
        time.sleep(2.0)
        iter = 0

    iter += 1
    

wikipedia_performance_df = pd.DataFrame(results)

Evaluating Search:   0%|          | 0/60 [00:00<?, ?it/s]

Evaluating Search: 100%|██████████| 60/60 [00:00<00:00, 74.52it/s]


In [57]:
wikipedia_performance_df

,name,summary,storyline,hr,mrr,type
0,0,0,2,0.116667,0.082500,lexical
1,0,0,2,0.083333,0.070833,hybrid
2,0,1,2,0.800000,0.734722,lexical
3,0,1,2,0.783333,0.717500,hybrid
4,0,2,2,0.800000,0.734722,lexical
5,0,2,2,0.783333,0.717500,hybrid
6,0,3,2,0.800000,0.734722,lexical
7,0,3,2,0.783333,0.717500,hybrid
8,1,0,2,0.150000,0.106111,lexical
9,1,0,2,0.133333,0.094167,hybrid


In [56]:
df = wikipedia_performance_df.sort_values("mrr", ascending=False)
print(df[df.type == "lexical"].head(1))
print()
print(df[df.type == "hybrid"].head(1))

   name  summary  storyline   hr       mrr     type
4     0        2          2  0.8  0.734722  lexical

    name  summary  storyline        hr     mrr    type
11     1        1          2  0.783333  0.7175  hybrid


### 🔧 <span style=color:silver>Tools and final RAG answer evaluation</span>

Here is the final evaluation. We want to see the performance of our agent when
using the entire RAG system and the optimized boost values. So we use two 
sources of evaluation. The user's evaluation and a separate llm-judge evaluation. 
The judge will evaluate the tool usage and the final answer quality based on the 
ground truth. The user only evaluates the final answer with good or bad review.

> This will take a very long time ⏰

In [ ]:
judge = RAGClient(opensearch_client, model="gemma-4-31b-it")
evaluator.evaluate_agent(judge)

## 📊 <span style=color:gold>Monitoring</span>

The final step is the monitoring, we must see the **usage** (tokens and cost) 
and **performance** (reviews) of our agent.  
For this, we can open the *streamlit* app in this address http://localhost:8501,
Click to the 